# Chapter 13 — Debug the Data Before the Model

**Book alignment:** Debugging AI From First Principles, Chapter 13

**Question this notebook isolates:** Validation AUC 0.99, live AUC 0.61. Does the
train-vs-data swap convict the data before anyone touches the model — **Run A** (same model
× de-leaked, honestly re-split data) collapses toward live, **Run B** (trivial baseline ×
the suspect data) stays near 0.99 — both arms agreeing on H2?

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

N = 800
y = rng.integers(0, 2, N)
x_legit = 0.6 * y + rng.normal(0, 1.0, N)          # weak, legitimate signal
x_leak  = y + rng.normal(0, 0.05, N)              # a post-event column: almost the label

def auc(score, label):
    score, label = np.asarray(score, float), np.asarray(label)
    pos, neg = score[label == 1], score[label == 0]
    return float((pos[:, None] > neg[None, :]).mean() + 0.5 * (pos[:, None] == neg[None, :]).mean())

idx = np.arange(N)
tr, va = idx[:600], idx[600:]
# split corruption: 15% of TRAIN rows copied into VALIDATION with fresh indices
leaked_va = np.concatenate([va, rng.choice(tr, size=90, replace=False)])

full_model     = lambda X: X @ np.array([1.0, 3.0])   # uses [x_legit, x_leak]
baseline_model = lambda X: X @ np.array([0.0, 1.0])   # x_leak alone (trivial)
X = np.column_stack([x_legit, x_leak])

## 1. The star result, and the leak audit

In [ ]:
auc_suspect = auc(full_model(X[leaked_va]), y[leaked_va])
overlap = len(set(leaked_va.tolist()) & set(tr.tolist())) / len(leaked_va)
print(f"suspect validation AUC : {auc_suspect:.3f}")
print(f"train∩validation overlap: {overlap:.0%}")
assert auc_suspect > 0.95 and overlap > 0.1
print("both a leaky column AND duplicated rows - two H2 defects, zero model changes so far")

## 2. Run A (same model × clean data) and Run B (baseline × suspect data)

In [ ]:
# Run A: drop the leak, rebuild the split honestly (no overlap), same model architecture
Xc = np.column_stack([x_legit, np.zeros(N)])          # x_leak quarantined
auc_A = auc(full_model(Xc[va]), y[va])

# Run B: trivial baseline on the untouched suspect data + split
auc_B = auc(baseline_model(X[leaked_va]), y[leaked_va])

print(f"Run A  same-model × clean+re-split : {auc_A:.3f}   (forecast: collapses toward live ~0.6)")
print(f"Run B  baseline  × suspect data    : {auc_B:.3f}   (forecast: stays ~0.99)")
assert auc_A < 0.75                                   # the 0.99 was leak-bought
assert auc_B > 0.95                                   # a one-feature model already 'wins'
print("\nboth arms agree -> H2 convicted. architecture tuning was never owed.")

## 3. The quarantine artifact (not a bigger network)

In [ ]:
QUARANTINE = {
    "dropped_columns": ["x_leak"],
    "split_rule": "temporal, no ID reuse; max(train.event_ts) < min(val.event_ts)",
    "regression_assertion": "assert set(train.order_id) & set(val.order_id) == set()",
}
for k, v in QUARANTINE.items():
    print(f"{k}: {v}")
# the honest number the model actually earns on legitimate signal:
print(f"\nhonest AUC on x_legit alone: {auc(x_legit[va], y[va]):.3f}  <- this is the real starting point")
assert 0.55 < auc(x_legit[va], y[va]) < 0.8

## What we earned

A validation score is a claim about the relationship between two datasets, wearing a model's
name. The swap probe convicted the data without a single retrain: Run A collapsed 0.99 →
~0.6 once the leaked column was quarantined and the split rebuilt without ID reuse, and Run
B showed a one-feature baseline already scoring ~0.99 on the suspect split. Both arms
agreeing is the conviction; either alone is a lead. The fix is a column quarantine plus a
split-integrity regression test.

**Notebook 14 / Chapter 14** moves downstream to the first thing that breaks once the data
is honest: tensor shapes, dtypes, and devices.